# Roblox Crowd Prediction

**Pertanyaan:** apakah kunjungan sepanjang masa cukup untuk memperkirakan keramaian saat snapshot diambil?

Notebook ini membaca dataset dan hasil eksperimen yang disertakan. Pelatihan lengkap dijalankan terpisah dengan `python src/train.py`. Sel Python di notebook ini sudah dijalankan saat paket dibuat. Tidak diperlukan akun Roblox.

In [1]:
from pathlib import Path
import json
import pandas as pd
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'roblox-crowd']
ROOT = next(p for p in candidates if (p / 'src/train.py').exists())
games = pd.read_csv(ROOT / 'data/processed/games.csv')
train = games.loc[games['split'].eq('train')].copy()
test = games.loc[games['split'].eq('test')].copy()
print('Total:', len(games), '| Belajar:', len(train), '| Uji:', len(test))

Total: 301 | Belajar: 240 | Uji: 61


## 1. Apa isi satu baris?

Satu baris adalah satu game pada waktu pengambilan. Nama dan ID hanya keterangan. `playing` merupakan jawaban asli; model tidak menerimanya sebagai input.

In [2]:
columns = ['name', 'visits', 'favorites', 'age_days', 'max_players', 'playing', 'crowd_level']
print(train[columns].head(5).to_string(index=False))
print('\nNilai kosong pada input:')
print(train[['visits', 'favorites', 'age_days', 'max_players']].isna().sum().to_string())

                              name      visits  favorites    age_days  max_players  playing crowd_level
                    Brookhaven 🏡RP 88573155840   30504528 2369.011356           30   302224       ramai
                     [🎃] Adopt Me! 45089101973   29759895 3380.295581           35   245688       ramai
                       Blox Fruits 64972425622   20080132 2829.960186           12   138181       ramai
              [💀] Steal a Brainrot 73975057573   29551543  518.071377            8    99628       ramai
[SKY ASSASSIN] Jujutsu Shenanigans  7624908417    3116779 1642.903321           20   116347       ramai

Nilai kosong pada input:
visits         0
favorites      0
age_days       0
max_players    0


## 2. EDA dari data belajar

Label: sepi untuk 0–99 pemain, sedang untuk 100–999, ramai untuk 1.000+. Batas ini merupakan keputusan eksperimen, bukan standar Roblox.

![Distribusi dan hubungan kunjungan dengan pemain](../results/training_eda.png)

Setiap titik pada panel kanan adalah satu game latih. Sumbu logaritmik merapatkan hitungan yang rentangnya sangat besar. Titik dengan kunjungan besar tidak selalu berada di kelas ramai.

In [3]:
print('Distribusi label pada data latih:')
print(train['crowd_level'].value_counts().to_string())
print('\nRingkasan statistik data latih:')
print(train[['visits', 'favorites', 'age_days', 'max_players']].describe().round(2).to_string())

Distribusi label pada data latih:
crowd_level
ramai     200
sedang     26
sepi       14

Ringkasan statistik data latih:
             visits    favorites  age_days  max_players
count  2.400000e+02       240.00    240.00       240.00
mean   2.659551e+09   2931223.22    827.88        18.64
std    9.975722e+09   8025753.93   1168.58        15.25
min    2.108740e+05       769.00     11.99         1.00
25%    1.147034e+07     91132.25     45.62         7.00
50%    7.999694e+07    429617.00    276.52        14.50
75%    6.080530e+08   2045090.50   1160.72        25.00
max    8.857316e+10  94839787.00   6775.21       120.00


## 3. Pastikan kreator terpisah

Game dari kreator yang sama dapat memiliki karakteristik mirip. Kita mengelompokkan berdasarkan jenis dan ID kreator, lalu memisahkannya antara belajar dan uji. ID kreator sendiri tidak menjadi fitur.

In [4]:
overlap = set(train['creator_group']) & set(test['creator_group'])
assert not overlap
print('Jumlah kreator yang muncul di kedua bagian:', len(overlap))
print('Empat lipatan validasi di dalam data latih:')
print(pd.read_json(ROOT / 'results/cv_folds.json').to_string(index=False))

Jumlah kreator yang muncul di kedua bagian: 0
Empat lipatan validasi di dalam data latih:
 fold  train  validation  creator_overlap
    1    180          60                0
    2    180          60                0
    3    180          60                0
    4    180          60                0


## 4. Bagaimana fitur disiapkan?

Dua kelompok fitur dibandingkan: kunjungan+umur, serta profil lengkap dengan favorit, kapasitas server dan rasio favorit per 1.000 kunjungan. Fitur deterministik dihitung tanpa memakai label. `log1p` dan standardisasi berada dalam Pipeline model; standardisasi dipelajari pada setiap lipatan data latih.

In [5]:
sample = train[['visits','favorites','age_days','max_players']].head(3).copy()
sample['favorites_per_1000_visits'] = 1000 * sample['favorites'] / sample['visits'].clip(lower=1)
assert 'playing' not in sample.columns
print(sample.round(3).to_string(index=False))

     visits  favorites  age_days  max_players  favorites_per_1000_visits
88573155840   30504528  2369.011           30                      0.344
45089101973   29759895  3380.296           35                      0.660
64972425622   20080132  2829.960           12                      0.309


## 5. Model mana yang dipilih?

Tabel ini berasal dari validasi data latih, bukan data uji. Angka tertinggi memilih kombinasi fitur dan model. Macro-F1 memberi bobot sama pada ketiga kategori; akurasi saja didominasi game ramai.

In [6]:
scores = pd.read_csv(ROOT / 'results/cv_results.csv')
print(scores[['feature_set','model','cv_macro_f1','cv_std','cv_accuracy']].round(4).to_string(index=False))

      feature_set               model  cv_macro_f1  cv_std  cv_accuracy
   profil_lengkap       random_forest       0.6330  0.0447       0.8583
riwayat_kunjungan       random_forest       0.5675  0.0371       0.8167
   profil_lengkap logistic_regression       0.5483  0.0869       0.7458
riwayat_kunjungan logistic_regression       0.4912  0.1113       0.7083
riwayat_kunjungan       decision_tree       0.4905  0.1007       0.6958
   profil_lengkap       decision_tree       0.4895  0.0515       0.6958
riwayat_kunjungan           mayoritas       0.3028  0.0102       0.8333
   profil_lengkap           mayoritas       0.3028  0.0102       0.8333


## 6. Evaluasi akhir

Model terpilih diuji setelah seluruh pemilihan selesai. Tebakan semua ramai merupakan pembanding yang ditetapkan sejak awal.

![Matriks kesalahan pada data uji](../results/confusion_matrix.png)

Baris adalah kelas asli, kolom adalah prediksi. Angka pada diagonal adalah prediksi benar.

In [7]:
metrics = json.loads((ROOT / 'results/test_metrics.json').read_text(encoding='utf-8'))
report = metrics['test_report']
baseline = metrics['majority_test_report']
print('Model:', metrics['selected_model'], '| Fitur:', metrics['feature_set'])
print('Accuracy model:', round(report['accuracy'], 4), '| pembanding:', round(baseline['accuracy'], 4))
print('Macro-F1 model:', round(report['macro avg']['f1-score'], 4), '| pembanding:', round(baseline['macro avg']['f1-score'], 4))
for label in ['sepi','sedang','ramai']:
    print(label, '| recall:', round(report[label]['recall'], 4), '| jumlah contoh:', int(report[label]['support']))
errors = pd.read_csv(ROOT / 'results/errors.csv')
print('\nSemua kesalahan uji:')
print(errors[['name','playing','crowd_level','prediction']].to_string(index=False))

Model: random_forest | Fitur: profil_lengkap
Accuracy model: 0.8361 | pembanding: 0.8197
Macro-F1 model: 0.4136 | pembanding: 0.3003
sepi | recall: 0.0 | jumlah contoh: 2
sedang | recall: 0.2222 | jumlah contoh: 9
ramai | recall: 0.98 | jumlah contoh: 50

Semua kesalahan uji:
                                             name  playing crowd_level prediction
Don’t Get Fired At Work! 🧳 [LIMITED SKIN EVENT! ]     3531       ramai     sedang
                           Murderers vs. Sheriffs      566      sedang      ramai
                     [UPD] 🏀 Basketball Training!      314      sedang      ramai
                                  Miami Streets 💥      243      sedang      ramai
                              Reborn As Swordsman      239      sedang      ramai
           Carpet Cleaning Simulator [HALLOWEEN!]       76        sepi      ramai
                Steals a Brainrot Trading Plaza 🛒      106      sedang      ramai
                                    Zombie Tycoon      112      sed

## 7. Kesimpulan yang dapat dipertanggungjawabkan

Model belajar lebih banyak daripada sekadar menebak kelas mayoritas, tetapi belum andal pada kelas kecil. Dua contoh sepi di data uji terlalu sedikit untuk kesimpulan luas. Profil lengkap mendapat CV macro-F1 lebih tinggi daripada kunjungan+umur pada eksperimen ini; tidak ada klaim sebab akibat.

Data merupakan katalog game yang dilacak, bukan seluruh Roblox. Snapshot ini tidak menguji ramalan besok atau game yang belum dirilis. Setelah melihat kesalahan uji, jangan menyetel model lalu tetap menyebut data yang sama sebagai evaluasi independen.

**Latihan memahami:** mengapa 81,97% dari menebak semua ramai belum cukup? Mengapa kapasitas satu server tidak sama dengan total pemain aktif? Mengapa kita menyimpan data uji sejak awal?